# LensIQ: deploy the camera-fog / lens-condition detector

Standalone PyFunc that flags fogged or contaminated camera lenses on a
frame-by-frame basis. Pure-Python (numpy + Pillow) - no external API,
no model weights, no GPU. Runs entirely inside the Databricks Model
Serving container.

Use case: camera-health diagnostics for QSR / c-store / forecourt
deployments. Dome cameras in cooler aisles fog from humidity; outdoor
PTZs accumulate condensation, dust, or rain droplets; smudged lenses
degrade silently and tank downstream model accuracy. This detector
outputs a `fogged` or `clear` verdict with a bbox around the affected
region so a store ops team can route a cleaning ticket.

Approach: patch-based Laplacian variance (high-frequency energy as a
sharpness proxy) gated on mean brightness (fog scatters light, shadows
don't). Connected fogged patches collapse into a single bbox via 4-way
flood fill so a localized smudge produces one detection instead of N.

Payload (matches the YOLO + Roboflow endpoints so the AppKit server's
`_normalizeDatabricks` works unchanged):

```json
{"dataframe_records": [{"image": "<b64>"}]}
```

Response:

```json
{"predictions": [[{"label": "fogged", "class_id": 1, "confidence": 0.94, "bbox": [320,240,639,479]}]]}
```

In [ ]:
dbutils.widgets.text("catalog", "iot_dev")
dbutils.widgets.text("schema", "lensiq")
dbutils.widgets.text("registered_name", "lensiq_fog_detector")
dbutils.widgets.text("endpoint_name", "lensiq-fog-detector")

In [ ]:
%pip install -q mlflow>=2.13 numpy>=1.26 "Pillow>=10.0"
dbutils.library.restartPython()

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
LOG = logging.getLogger("deploy_fog_detector")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
REGISTERED = f"{CATALOG}.{SCHEMA}.{dbutils.widgets.get('registered_name')}"
ENDPOINT = dbutils.widgets.get("endpoint_name")

LOG.info("Deploying fog detector -> %s", ENDPOINT)
LOG.info("  registered_model=%s", REGISTERED)

## PyFunc wrapper

Patch-grid Laplacian-variance + brightness classifier. Thresholds are
calibrated on the synthesized partial-fog clips shipped under
`client/public/sample-videos/`. See the class docstring for details.

In [ ]:
import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.models import infer_signature


class FogDetector(mlflow.pyfunc.PythonModel):
    """Patch-based camera-fog / lens-condition classifier.

    Pipeline (per frame):
      1. Decode base64 image + convert to grayscale (Pillow).
      2. Tile into a `PATCH_GRID_ROWS x PATCH_GRID_COLS` grid.
      3. Per patch, compute:
           - Laplacian variance (high-frequency energy, low = blurry/foggy)
           - mean brightness          (fog scatters light, shadow doesn't)
      4. Flag a patch as `fogged` iff BOTH:
           - sharpness (log10 lap-var) < PATCH_FOG_PIVOT
           - brightness              >= PATCH_BRIGHTNESS_FLOOR
         The brightness gate eliminates the most common false positive:
         dark corners on a clear scene also have low Laplacian variance,
         but they're not fogged.
      5. 4-way flood-fill connected fogged patches into bboxes so a
         localized smudge becomes ONE detection instead of N adjacent ones.

    Always returns at least one detection:
      - No fogged patches  -> single full-frame `clear` verdict.
      - Otherwise          -> one `fogged` bbox per connected region.

    Output schema matches the YOLO + Roboflow detectors:
      `{label, class_id, confidence, bbox: [x1, y1, x2, y2]}`
    so the AppKit server's `_normalizeDatabricks` consumes it unchanged.
    """

    # Tuning. Calibrated against the synthesized cstore-foggy-lens and
    # forecourt-foggy-lens clips (radial center-blob Gaussian fog).
    # Clear patches sit at sharpness 2.5-3.4 (lap_var 300-2500);
    # heavily fogged patches collapse to 0.3-1.4 (lap_var 2-25).
    FOG_SHARPNESS_PIVOT = 2.0
    FOG_SIGMOID_SLOPE = 3.0
    PATCH_GRID_COLS = 8
    PATCH_GRID_ROWS = 6
    PATCH_FOG_PIVOT = 1.4
    PATCH_BRIGHTNESS_FLOOR = 80

    def _strip_data_url(self, image_b64):
        if isinstance(image_b64, str) and image_b64.startswith("data:"):
            return image_b64.split(",", 1)[1]
        return image_b64 or ""

    def _decode_to_grayscale(self, image_b64):
        """Decode a base64 JPEG/PNG into a numpy grayscale uint8 array.
        Returns (arr, width, height) or (None, 0, 0) on failure."""
        import base64
        import io
        import numpy as np
        from PIL import Image
        try:
            raw = base64.b64decode(self._strip_data_url(image_b64))
            im = Image.open(io.BytesIO(raw)).convert("L")
            arr = np.asarray(im, dtype=np.uint8)
            return arr, im.width, im.height
        except Exception:
            return None, 0, 0

    def _laplacian_variance(self, arr):
        """Variance of a 4-neighbor Laplacian over the patch. Low values
        indicate a blurry / fogged / out-of-focus frame."""
        import numpy as np
        a = arr.astype(np.float32)
        lap = (
            np.roll(a, -1, 0) + np.roll(a, 1, 0)
            + np.roll(a, -1, 1) + np.roll(a, 1, 1)
            - 4 * a
        )
        # Strip the 1-pixel border to drop wrap-around artifacts from np.roll.
        return float(lap[1:-1, 1:-1].var())

    def _connected_components(self, fog_grid):
        """4-connected flood fill over a 2D bool grid. Iterative so we keep
        numpy/Pillow as the only deps (no scipy.ndimage)."""
        import numpy as np
        rows, cols = fog_grid.shape
        visited = np.zeros_like(fog_grid)
        components = []
        for r0 in range(rows):
            for c0 in range(cols):
                if not fog_grid[r0, c0] or visited[r0, c0]:
                    continue
                comp = []
                stack = [(r0, c0)]
                while stack:
                    r, c = stack.pop()
                    if visited[r, c]:
                        continue
                    visited[r, c] = True
                    if not fog_grid[r, c]:
                        continue
                    comp.append((r, c))
                    if r > 0:
                        stack.append((r - 1, c))
                    if r < rows - 1:
                        stack.append((r + 1, c))
                    if c > 0:
                        stack.append((r, c - 1))
                    if c < cols - 1:
                        stack.append((r, c + 1))
                if comp:
                    components.append(comp)
        return components

    def _detect(self, image_b64):
        import math
        import numpy as np
        arr, w, h = self._decode_to_grayscale(image_b64)
        if arr is None or w == 0 or h == 0:
            return [{"label": "error", "class_id": -1, "confidence": 0.0,
                     "bbox": [0, 0, 0, 0],
                     "error": "fog-detector: failed to decode image"}]

        cols, rows = self.PATCH_GRID_COLS, self.PATCH_GRID_ROWS
        pw = max(1, w // cols)
        ph = max(1, h // rows)

        sharp_grid = np.zeros((rows, cols), dtype=np.float32)
        bright_grid = np.zeros((rows, cols), dtype=np.float32)
        for r in range(rows):
            for c in range(cols):
                patch = arr[r * ph:(r + 1) * ph, c * pw:(c + 1) * pw]
                if patch.size == 0:
                    continue
                sharp_grid[r, c] = math.log10(max(1.0, self._laplacian_variance(patch)))
                bright_grid[r, c] = float(patch.mean())

        fog_grid = (sharp_grid < self.PATCH_FOG_PIVOT) & (bright_grid >= self.PATCH_BRIGHTNESS_FLOOR)

        if not fog_grid.any():
            # Whole-frame clear verdict.
            mean_sharp = float(sharp_grid.mean())
            p_clear = 1.0 / (1.0 + math.exp(
                -self.FOG_SIGMOID_SLOPE * (mean_sharp - self.FOG_SHARPNESS_PIVOT)
            ))
            return [{
                "label": "clear",
                "class_id": 0,
                "confidence": float(p_clear),
                "bbox": [0, 0, int(w - 1), int(h - 1)],
            }]

        components = self._connected_components(fog_grid)
        detections = []
        for comp in components:
            rs = [r for r, _ in comp]
            cs = [c for _, c in comp]
            x1 = min(cs) * pw
            y1 = min(rs) * ph
            x2 = min((max(cs) + 1) * pw, w) - 1
            y2 = min((max(rs) + 1) * ph, h) - 1
            avg_sharp = float(np.mean([sharp_grid[r, c] for r, c in comp]))
            p_clear = 1.0 / (1.0 + math.exp(
                -self.FOG_SIGMOID_SLOPE * (avg_sharp - self.FOG_SHARPNESS_PIVOT)
            ))
            detections.append({
                "label": "fogged",
                "class_id": 1,
                "confidence": float(1.0 - p_clear),
                "bbox": [int(x1), int(y1), int(x2), int(y2)],
            })
        return detections

    def predict(self, context, model_input, params=None):
        if hasattr(model_input, "to_dict"):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)
        return [self._detect(r.get("image")) for r in rows]

## Log + register

In [ ]:
_TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAH"
    "ggJ/PchI7wAAAABJRU5ErkJggg=="
)

sample_input = pd.DataFrame([{"image": _TINY_PNG_B64}])
sample_output = [[]]
signature = infer_signature(sample_input, sample_output)

mlflow.set_registry_uri("databricks-uc")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

with mlflow.start_run(run_name="deploy_fog_detector") as run:
    info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=FogDetector(),
        signature=signature,
        input_example=sample_input,
        registered_model_name=REGISTERED,
        pip_requirements=[
            "mlflow>=2.13",
            "numpy>=1.26",
            "Pillow>=10.0",
        ],
    )
    mlflow.set_tag("lensiq.detector_slug", "fog_detector")
    mlflow.set_tag("lensiq.display_name", "Camera fog / lens condition")
LOG.info("Logged model URI: %s", info.model_uri)

## Create / update the serving endpoint

No environment variables to inject - the fog detector has no external
dependencies and no secrets.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

client = mlflow.MlflowClient()
versions = client.search_model_versions(f"name='{REGISTERED}'")
latest_version = max(versions, key=lambda v: int(v.version)).version
LOG.info("Deploying %s version %s -> endpoint %s", REGISTERED, latest_version, ENDPOINT)

served = ServedEntityInput(
    entity_name=REGISTERED,
    entity_version=latest_version,
    workload_size="Small",
    scale_to_zero_enabled=True,
)

w = WorkspaceClient()
try:
    w.serving_endpoints.get(name=ENDPOINT)
    LOG.info("Endpoint exists; updating config")
    w.serving_endpoints.update_config(name=ENDPOINT, served_entities=[served])
except Exception:
    LOG.info("Endpoint not found; creating")
    w.serving_endpoints.create(
        name=ENDPOINT,
        config=EndpointCoreConfigInput(name=ENDPOINT, served_entities=[served]),
    )
LOG.info("Submitted deployment for %s; watch Serving UI for readiness.", ENDPOINT)